In [ ]:
import os
import h5py
import numpy as np
import multiprocessing as mp
from glob import glob
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
import warnings
from Bio.PDB.PDBExceptions import PDBConstructionWarning
warnings.simplefilter('ignore', PDBConstructionWarning)

In [4]:
# 氨基酸字母表（'-' 代表未知或缺失）
PDB_AA = '-ACDEFGHIKLMNPQRSTVWY'
# DSSP 二级结构类型
# - : unknown
# H : α-helix
# B : β-bridge
# E : β-sheet
# G : 3-helix
# I : π-helix
# T : turn
# S : bend
DSSP_SS = '-HBEGITS'
# 氢键能量阈值（DSSP 中小于该值认为存在氢键）
HBOND_CUTOFF = -0.5

In [ ]:
def pdb_worker(ifn):
    """
    解析 PDB 文件，提取：
    - 节点特征（氨基酸 / 二级结构 / RSA）
    - 原子坐标（N, CA, C, O, CB/伪CB）
    - 肽键（peptide bond）
    - 氢键（hydrogen bond）
    并保存为 HDF5
    """

    ofn = ifn.replace('.pdb', '.hdf5')
    # 若已处理则跳过
    if os.path.exists(ofn):
        return('already exists')
    # 解析 PDB 与 DSSP
    try:
        pdb = PDBParser().get_structure('NULL', ifn)[0]  # 只取第一个 model
        dssp = DSSP(pdb, ifn, dssp='mkdssp')
    except:
        return('PDB/DSSP parse error')
    # 初始化容器
    node  = []   # 节点特征
    coord = []   # 原子坐标
    pbond = []   # 肽键边
    hbond = []   # 氢键边
    idx   = {}   # DSSP residue index -> 节点 index
    # 遍历残基与 DSSP 结果
    for i, j in zip(pdb.get_residues(), dssp):
        atom = sorted(list(i.get_atoms()))
        try:
            # 确保主链原子顺序完整
            if atom[0].get_name() != 'N':  continue
            if atom[1].get_name() != 'CA': continue
            if atom[2].get_name() != 'C':  continue
            if atom[3].get_name() != 'O':  continue
            # 非 GLY 必须有 CB
            if i.get_resname() != 'GLY' and atom[4].get_name() != 'CB':
                continue
            # DSSP 无效残基
            if j[3] == 'NA':
                continue
        except:
            continue
        # 建立 DSSP residue index 到节点编号的映射
        idx[j[0] + 0] = len(node)
        # 节点特征
        aa  = max(PDB_AA.find(j[1]), 0)   # 氨基酸类型
        ss  = max(DSSP_SS.find(j[2]), 0)  # 二级结构
        rsa = j[3] + 0                    # 相对溶剂可及性
        node.append([aa, ss, rsa])
        # 原子坐标（N, CA, C, O, CB）
        c  = [a.get_coord() for a in atom[:5]]
        # 若缺失 CB，则用虚拟 CB
        c0 = c[0] - c[1]; c0 = c0 / np.sum(c0**2)**.5
        c2 = c[2] - c[1]; c2 = c2 / np.sum(c2**2)**.5
        h  = c[1] - (c0 + c2)
        if len(c) < 5:
            c.append(h)
        elif atom[4].get_name() != 'CB':
            c[4] = h
        coord.append(c)
        # 肽键（相邻残基）
        if j[4] < 360 and j[0] - 1 in idx:
            pbond.append([len(node) - 2, len(node) - 1])
        # DSSP 氢键（最多 4 种）
        if j[6]  != 0 and j[7]  <= HBOND_CUTOFF:
            hbond.append([j[0]+0, j[0]+j[6]])
        if j[8]  != 0 and j[9]  <= HBOND_CUTOFF:
            hbond.append([j[0]+j[8], j[0]+0])
        if j[10] != 0 and j[11] <= HBOND_CUTOFF:
            hbond.append([j[0]+0, j[0]+j[10]])
        if j[12] != 0 and j[13] <= HBOND_CUTOFF:
            hbond.append([j[0]+j[12], j[0]+0])
    # 转为 numpy
    node = np.array(node)
    if node.size == 0:
        return('no valid residues')
    coord = np.array(coord)
    if len(node) != len(coord):
        return('residue/coord size mismatch')
    # 肽键边 [2, E]
    pbond = np.array(pbond).T
    if pbond.size == 0:
        return('no peptide bonds')
    # DSSP residue index -> 节点 index
    hbond = [[idx.get(j, -1) for j in i] for i in hbond]
    hbond = np.unique(np.array(hbond), axis=0)
    hbond = hbond[np.all(hbond >= 0, axis=-1)].T
    # 按终点排序
    hbond = hbond[:, np.argsort(hbond[1], kind='stable')] \
        if hbond.size else np.zeros([2, 0])
    # 保存 HDF5
    with h5py.File(ofn, 'w') as f:
        f.create_dataset('node_aa',  data=node[:, 0].astype(np.int8))
        f.create_dataset('node_ss',  data=node[:, 1].astype(np.int8))
        f.create_dataset('node_rsa', data=node[:, 2].astype(np.float16))
        f.create_dataset('node_pos', data=coord.astype(np.float16))
        f.create_dataset('edge_pep', data=pbond.astype(np.int16))
        f.create_dataset('edge_nho', data=hbond.astype(np.int16))

In [ ]:
with mp.Pool(os.cpu_count() // 2) as p:
    # PDB 结构
    p.map(pdb_worker, glob('/home/burger/data/pdb/*/*.pdb'))

In [27]:
missing_hdf5 = []
root_dir = "/home/burger/data/pdb"

for root, dirs, files in os.walk(root_dir):
    pdb_files = [f for f in files if f.endswith(".pdb")]
    hdf5_files = {f for f in files if f.endswith(".hdf5")}

    for pdb in pdb_files:
        hdf5_name = os.path.splitext(pdb)[0] + ".hdf5"
        if hdf5_name not in hdf5_files:
            pdb_path = os.path.join(root, pdb)
            missing_hdf5.append(pdb_path)

for file in missing_hdf5:
    result = pdb_worker(file)
    print(f"Processed {file}: {result}")

Processed /home/burger/data/pdb/TC/6TCH-A.pdb: no peptide bonds
Processed /home/burger/data/pdb/Y1/1Y1V-S.pdb: no valid residues
Processed /home/burger/data/pdb/OR/1OR8-B.pdb: no valid residues
Processed /home/burger/data/pdb/VO/5VOX-e.pdb: no peptide bonds
Processed /home/burger/data/pdb/JB/5JBQ-B.pdb: no valid residues
Processed /home/burger/data/pdb/JP/2JPX-A.pdb: no valid residues
Processed /home/burger/data/pdb/F4/6F4P-B.pdb: no peptide bonds
Processed /home/burger/data/pdb/F4/7F4U-C.pdb: no valid residues
Processed /home/burger/data/pdb/GA/5GAR-I.pdb: no peptide bonds
Processed /home/burger/data/pdb/DJ/6DJK-B.pdb: no peptide bonds
Processed /home/burger/data/pdb/TK/7TKO-U.pdb: no peptide bonds
Processed /home/burger/data/pdb/TK/1TKQ-A.pdb: no peptide bonds
Processed /home/burger/data/pdb/UQ/4UQ8-U.pdb: no peptide bonds
Processed /home/burger/data/pdb/LK/2LK9-A.pdb: no peptide bonds


/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)


Processed /home/burger/data/pdb/HX/6HX3-B.pdb: PDB/DSSP parse error
Processed /home/burger/data/pdb/Q0/6Q0B-7.pdb: no peptide bonds
Processed /home/burger/data/pdb/PJ/1PJD-A.pdb: no valid residues
Processed /home/burger/data/pdb/Q1/7Q1V-D.pdb: no peptide bonds
Processed /home/burger/data/pdb/FJ/1FJA-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/PZ/7PZN-M.pdb: no valid residues
Processed /home/burger/data/pdb/U2/3U2Q-B.pdb: no valid residues
Processed /home/burger/data/pdb/TJ/7TJZ-X.pdb: no peptide bonds
Processed /home/burger/data/pdb/OL/7OLE-H.pdb: no peptide bonds
Processed /home/burger/data/pdb/OL/7OLE-J.pdb: no peptide bonds
Processed /home/burger/data/pdb/A3/1A37-P.pdb: no valid residues
Processed /home/burger/data/pdb/AR/5ARA-U.pdb: no peptide bonds
Processed /home/burger/data/pdb/AR/5ARA-T.pdb: no peptide bonds
Processed /home/burger/data/pdb/OP/7OPC-c.pdb: no peptide bonds
Processed /home/burger/data/pdb/DE/8DEV-D.pdb: no peptide bonds
Processed /home/burger/data/pdb/

/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)


Processed /home/burger/data/pdb/DL/3DL8-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/TI/1TIA-A.pdb: no valid residues
Processed /home/burger/data/pdb/ZJ/2ZJP-5.pdb: no valid residues
Processed /home/burger/data/pdb/L3/3L37-H.pdb: no peptide bonds
Processed /home/burger/data/pdb/CW/1CWM-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/CW/1CWH-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/CW/1CWO-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/CW/1CWI-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/CW/1CWF-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/G3/3G3P-D.pdb: no peptide bonds
Processed /home/burger/data/pdb/D8/1D8T-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/HS/5HSV-E.pdb: no peptide bonds
Processed /home/burger/data/pdb/PG/4PGC-H.pdb: no peptide bonds
Processed /home/burger/data/pdb/Q6/7Q6I-X.pdb: no valid residues
Processed /home/burger/data/pdb/E5/5E5T-B.pdb: no peptide bonds
Processed /home/burger/data/pdb/E5/6E

/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)
/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)


Processed /home/burger/data/pdb/LB/7LB6-M.pdb: PDB/DSSP parse error
Processed /home/burger/data/pdb/HY/6HYE-B.pdb: PDB/DSSP parse error
Processed /home/burger/data/pdb/Q2/7Q21-x.pdb: no peptide bonds
Processed /home/burger/data/pdb/Q2/7Q21-X.pdb: no peptide bonds
Processed /home/burger/data/pdb/TG/5TGL-A.pdb: no valid residues
Processed /home/burger/data/pdb/TG/1TGL-A.pdb: no valid residues
Processed /home/burger/data/pdb/NE/5NES-E.pdb: no peptide bonds
Processed /home/burger/data/pdb/C4/1C4B-A.pdb: no peptide bonds
Processed /home/burger/data/pdb/PA/6PAT-B.pdb: no peptide bonds
Processed /home/burger/data/pdb/PA/6PAT-A.pdb: no peptide bonds
Processed /home/burger/data/pdb/OB/7OB8-B.pdb: no peptide bonds
Processed /home/burger/data/pdb/KQ/1KQE-A.pdb: no peptide bonds
Processed /home/burger/data/pdb/EE/6EEN-D.pdb: PDB/DSSP parse error


/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)
/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)
/home/burger/miniconda3/envs/cu121/lib/python3.11/site-packages/Bio/PDB/DSSP.py:199: UserWarning: DSSP could not be created due to an error:
empty protein, or no valid complete residues

  warnings.warn(err)


Processed /home/burger/data/pdb/EE/6EEN-C.pdb: PDB/DSSP parse error
Processed /home/burger/data/pdb/EE/6EEN-B.pdb: PDB/DSSP parse error
Processed /home/burger/data/pdb/OD/7OD8-E.pdb: no valid residues
Processed /home/burger/data/pdb/OD/7OD6-E.pdb: no valid residues
Processed /home/burger/data/pdb/OD/7OD7-E.pdb: no valid residues
Processed /home/burger/data/pdb/Q3/2Q3I-D.pdb: no peptide bonds
Processed /home/burger/data/pdb/BC/1BCK-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/W2/6W2U-E.pdb: no peptide bonds
Processed /home/burger/data/pdb/RC/7RC6-C.pdb: no peptide bonds
Processed /home/burger/data/pdb/PB/6PBS-D.pdb: no peptide bonds
Processed /home/burger/data/pdb/GT/5GTR-C.pdb: no peptide bonds


In [25]:
from pathlib import Path

def count_files_pathlib(directory_path, recursive=True):
    path = Path(directory_path)
    if not path.exists():
        return "路径不存在"
    if recursive:
        # rglob('*') 会递归查找所有内容
        # is_file() 确保只计算文件，不计算文件夹
        count = sum(1 for file in path.rglob('*') if file.is_file())
        mode = "（包含子文件夹）"
    else:
        # iterdir() 只遍历当前层级
        count = sum(1 for file in path.iterdir() if file.is_file())
        mode = "（仅当前层级）" 
    return count, mode

# --- 使用示例 ---
target_folder = r"../data/pdb"  # 替换为你的目标文件夹路径

# 计算
total, mode_text = count_files_pathlib(target_folder, recursive=True) # 改为 False 则不查子文件夹
print(f"文件夹下{mode_text}共有: {total} 个文件")

文件夹下（包含子文件夹）共有: 214133 个文件


In [29]:
import h5py
import numpy as np
from glob import glob


hdf5_files = glob('/home/burger/data/pdb/*/*.hdf5')
node_seq, node_pos, node_idx = [], [], [0]
edge_nho, edge_idx = [], [0]
label = []
dt = h5py.string_dtype(encoding='utf-8')

        
for fn in hdf5_files:
    try:
        with h5py.File(fn, 'r') as data:
            node_seq.append(data['node_aa'][()])
            node_pos.append(data['node_pos'][()])
            assert len(node_seq[-1]) == len(node_pos[-1])
            node_idx.append(len(node_seq[-1]))
            edge_nho.append(data['edge_nho'][()].T)
            edge_idx.append(len(edge_nho[-1]))
            file_name = fn.split('/')[-1].replace('.hdf5', '')
            label.append(file_name)
    except: 
        print(f'Error reading {fn}, skipping.')
        continue
node_seq = np.concatenate(node_seq).astype(np.int8)
node_pos = np.concatenate(node_pos).astype(np.float16)
node_idx = np.cumsum(node_idx).astype(np.int64)
edge_nho = np.concatenate(edge_nho).astype(np.int16).T
edge_idx = np.cumsum(edge_idx).astype(np.int64)
label = np.array(label).astype(dt)

with h5py.File('/home/burger/bioinfo/project/pdb.hdf5', 'w') as f:
    f.create_dataset('node_seq', data=node_seq)
    f.create_dataset('node_pos', data=node_pos)
    f.create_dataset('node_idx', data=node_idx)
    f.create_dataset('edge_nho', data=edge_nho)
    f.create_dataset('edge_idx', data=edge_idx)
    f.create_dataset('label', data=label)
print('#done!!!')

#done!!!
